# Sistema de Registro de Gastos — Flask + SQLite

ESTUDIANTE: JUAN CCOPA GALLEGOS. La aplicación tendrá acceso con `admin` / `123`.

In [30]:
%pip -q install flask

In [31]:
import os, pathlib
os.makedirs('templates', exist_ok=True)
os.makedirs('static', exist_ok=True)


In [32]:
%%writefile app.py
import os, sqlite3, csv, io
from datetime import date
from functools import wraps
from flask import Flask, render_template, request, redirect, url_for, session, flash, Response
from werkzeug.security import generate_password_hash, check_password_hash

app = Flask(__name__)
app.secret_key = os.environ.get("SECRET_KEY", "cambia-esta-clave-en-produccion")

DB_PATH = os.environ.get("DB_PATH", "gastos.db")
# Credenciales solicitadas
USUARIO = "admin"
CLAVE_HASH = generate_password_hash("123")

def conectar():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn

def inicializar_db():
    with conectar() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS gastos (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                fecha TEXT NOT NULL,
                descripcion TEXT NOT NULL,
                monto REAL NOT NULL CHECK(monto >= 0)
            )
        """)

def login_requerido(fn):
    @wraps(fn)
    def wrapper(*args, **kwargs):
        if not session.get("autenticado"):
            return redirect(url_for("login"))
        return fn(*args, **kwargs)
    return wrapper

@app.route("/", methods=["GET"])
def inicio():
    return redirect(url_for("panel") if session.get("autenticado") else url_for("login"))

@app.route("/login", methods=["GET", "POST"])
def login():
    if request.method == "POST":
        usuario = request.form.get("usuario", "").strip()
        clave = request.form.get("clave", "")
        if usuario == USUARIO and check_password_hash(CLAVE_HASH, clave):
            session.clear()
            session["autenticado"] = True
            return redirect(url_for("panel"))
        flash("Usuario o contraseña incorrectos.", "error")
    return render_template("login.html")

@app.route("/salir")
def salir():
    session.clear()
    return redirect(url_for("login"))

@app.route("/panel", methods=["GET", "POST"])
@login_requerido
def panel():
    if request.method == "POST":
        fecha = request.form.get("fecha", "").strip()
        descripcion = request.form.get("descripcion", "").strip()
        monto_txt = request.form.get("monto", "").strip().replace(",", ".")
        try:
            monto = float(monto_txt)
            if not fecha or not descripcion or monto < 0:
                raise ValueError()
            date.fromisoformat(fecha)
            with conectar() as conn:
                conn.execute(
                    "INSERT INTO gastos (fecha, descripcion, monto) VALUES (?, ?, ?)",
                    (fecha, descripcion, monto)
                )
            flash("Gasto registrado correctamente.", "ok")
        except (ValueError, TypeError):
            flash("Verifica la fecha, descripción y monto ingresados.", "error")
        return redirect(url_for("panel"))

    desde = request.args.get("desde", "").strip()
    hasta = request.args.get("hasta", "").strip()
    sql = "SELECT * FROM gastos WHERE 1=1"
    params = []
    if desde:
        sql += " AND fecha >= ?"; params.append(desde)
    if hasta:
        sql += " AND fecha <= ?"; params.append(hasta)
    sql += " ORDER BY fecha DESC, id DESC"
    with conectar() as conn:
        gastos = conn.execute(sql, params).fetchall()
    total = sum(float(g["monto"]) for g in gastos)
    return render_template("panel.html", gastos=gastos, total=total,
                           desde=desde, hasta=hasta, hoy=date.today().isoformat())

@app.route("/eliminar/<int:gasto_id>", methods=["POST"])
@login_requerido
def eliminar(gasto_id):
    with conectar() as conn:
        conn.execute("DELETE FROM gastos WHERE id = ?", (gasto_id,))
    flash("Gasto eliminado.", "ok")
    return redirect(url_for("panel"))

@app.route("/reporte.csv")
@login_requerido
def reporte_csv():
    desde = request.args.get("desde", "").strip()
    hasta = request.args.get("hasta", "").strip()
    sql = "SELECT fecha, descripcion, monto FROM gastos WHERE 1=1"
    params = []
    if desde:
        sql += " AND fecha >= ?"; params.append(desde)
    if hasta:
        sql += " AND fecha <= ?"; params.append(hasta)
    sql += " ORDER BY fecha DESC"
    with conectar() as conn:
        filas = conn.execute(sql, params).fetchall()
    salida = io.StringIO()
    escritor = csv.writer(salida)
    escritor.writerow(["Fecha", "Descripción", "Monto"])
    for fila in filas:
        escritor.writerow([fila["fecha"], fila["descripcion"], f'{fila["monto"]:.2f}'])
    return Response("\ufeff" + salida.getvalue(), mimetype="text/csv",
                    headers={"Content-Disposition": "attachment; filename=reporte_gastos.csv"})

if __name__ == "__main__":
    inicializar_db()
    app.run(host="0.0.0.0", port=5000, debug=False)

Overwriting app.py


In [33]:
%%writefile templates/login.html
<!doctype html>
<html lang="es"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">
<title>Acceso | Mis Gastos</title><link rel="stylesheet" href="{{ url_for('static', filename='estilos.css') }}"></head>
<body class="login-bg"><main class="login-card">
<div class="marca">MG</div><h1>Mis Gastos</h1><p class="muted">Ingresa para administrar tus gastos</p>
{% with mensajes = get_flashed_messages(with_categories=true) %}{% for categoria, mensaje in mensajes %}
<div class="alert {{ categoria }}">{{ mensaje }}</div>{% endfor %}{% endwith %}
<form method="post"><label>Usuario</label><input name="usuario" required placeholder="Tu usuario" autocomplete="username">
<label>Contraseña</label><input type="password" name="clave" required placeholder="Tu contraseña" autocomplete="current-password">
<button class="btn full">Ingresar</button></form>


Overwriting templates/login.html


In [34]:
%%writefile templates/panel.html
<!doctype html>
<html lang="es"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">
<title>Panel | Mis Gastos</title><link rel="stylesheet" href="{{ url_for('static', filename='estilos.css') }}"></head>
<body><header class="topbar"><div><span class="marca small">MG</span><strong>Mis Gastos</strong></div><a href="{{ url_for('salir') }}" class="salir">Cerrar sesión ↗</a></header>
<main class="container"><section class="encabezado"><div><p class="eyebrow">PANEL DE CONTROL</p><h1>Registro de gastos</h1><p class="muted">Administra tus movimientos y consulta tus reportes.</p></div></section>
{% with mensajes = get_flashed_messages(with_categories=true) %}{% for categoria, mensaje in mensajes %}
<div class="alert {{ categoria }}">{{ mensaje }}</div>{% endfor %}{% endwith %}
<div class="resumen"><div><span>Total del periodo</span><strong>S/ {{ "%.2f"|format(total) }}</strong></div><div><span>Gastos registrados</span><strong>{{ gastos|length }}</strong></div></div>
<div class="layout"><section class="card"><h2>Registrar nuevo gasto</h2><form method="post" class="form-grid">
<div><label>Fecha</label><input type="date" name="fecha" value="{{ hoy }}" required></div>
<div><label>Monto (S/)</label><input type="number" name="monto" min="0" step="0.01" placeholder="0.00" required></div>
<div class="wide"><label>Descripción</label><input name="descripcion" maxlength="180" placeholder="Ej. Compra de alimentos" required></div>
<div class="wide"><button class="btn">Guardar gasto</button></div></form></section>
<section class="card"><h2>Generar reporte</h2><form method="get" class="filtros">
<div><label>Desde</label><input type="date" name="desde" value="{{ desde }}"></div>
<div><label>Hasta</label><input type="date" name="hasta" value="{{ hasta }}"></div>
<button class="btn">Filtrar</button><a class="btn secondary" href="{{ url_for('reporte_csv', desde=desde, hasta=hasta) }}">Descargar CSV</a>
</form><p class="hint">El reporte incluye los gastos del rango seleccionado.</p></section></div>
<section class="card tabla-card"><div class="tabla-head"><h2>Historial de gastos</h2><span>{{ gastos|length }} registro(s)</span></div>
{% if gastos %}<div class="table-wrap"><table><thead><tr><th>Fecha</th><th>Descripción</th><th class="right">Monto</th><th></th></tr></thead><tbody>
{% for gasto in gastos %}<tr><td>{{ gasto.fecha }}</td><td>{{ gasto.descripcion }}</td><td class="right">S/ {{ "%.2f"|format(gasto.monto) }}</td>
<td class="right"><form method="post" action="{{ url_for('eliminar', gasto_id=gasto.id) }}" onsubmit="return confirm('¿Eliminar este gasto?')"><button class="delete">Eliminar</button></form></td></tr>{% endfor %}
</tbody></table></div>{% else %}<p class="empty">Todavía no hay gastos registrados para este filtro.</p>{% endif %}</section>
<footer>Aplicación de registro personal de gastos · Flask + SQLite</footer></main></body></html>

Overwriting templates/panel.html


In [35]:
%%writefile static/estilos.css
*{box-sizing:border-box}body{margin:0;background:#f4f6fb;color:#1f2937;font-family:Arial,sans-serif}.login-bg{min-height:100vh;display:grid;place-items:center;background:linear-gradient(135deg,#e9efff,#f8faff)}.login-card{width:min(420px,92vw);background:#fff;padding:38px;border-radius:18px;box-shadow:0 18px 55px #263b6818;text-align:center}.marca{display:inline-grid;place-items:center;width:52px;height:52px;border-radius:15px;background:#315eea;color:white;font-weight:800;font-size:20px}.marca.small{width:34px;height:34px;border-radius:10px;font-size:13px;margin-right:10px}h1{margin:16px 0 8px;font-size:28px}h2{font-size:18px;margin:0 0 20px}.muted,.hint{color:#7b8497;font-size:14px}.hint{font-size:12px;margin-top:18px}form{text-align:left}label{display:block;font-size:13px;font-weight:700;margin:15px 0 7px}input{width:100%;border:1px solid #dce1eb;border-radius:9px;padding:12px;font-size:14px;background:#fff;color:#1f2937}input:focus{outline:2px solid #cbd7ff;border-color:#315eea}.btn{display:inline-block;border:0;border-radius:9px;background:#315eea;color:#fff;padding:12px 17px;font-weight:700;text-decoration:none;cursor:pointer;font-size:13px}.btn.full{width:100%;margin-top:22px}.btn.secondary{background:#eef2ff;color:#315eea}.topbar{height:68px;background:#fff;border-bottom:1px solid #e8ebf2;display:flex;align-items:center;justify-content:space-between;padding:0 max(calc((100vw - 1100px)/2),24px)}.topbar>div{display:flex;align-items:center}.salir{color:#536078;text-decoration:none;font-size:13px}.container{max-width:1100px;margin:38px auto;padding:0 22px}.encabezado{margin-bottom:24px}.eyebrow{color:#315eea;font-size:11px;font-weight:800;letter-spacing:1.5px;margin:0}.encabezado h1{margin:8px 0}.resumen{display:grid;grid-template-columns:repeat(2,1fr);gap:16px;margin-bottom:20px}.resumen>div,.card{background:#fff;border:1px solid #e9ecf3;border-radius:14px;padding:22px}.resumen span{display:block;color:#7b8497;font-size:13px;margin-bottom:8px}.resumen strong{font-size:25px}.layout{display:grid;grid-template-columns:1.2fr 1fr;gap:18px;margin-bottom:18px}.form-grid{display:grid;grid-template-columns:1fr 1fr;gap:0 14px}.wide{grid-column:1/-1}.form-grid .btn{margin-top:17px}.filtros{display:grid;grid-template-columns:1fr 1fr;gap:0 12px}.filtros .btn{margin-top:16px;text-align:center}.alert{padding:12px 15px;border-radius:9px;margin:12px 0;font-size:13px}.alert.ok{background:#e9f8ef;color:#187544}.alert.error{background:#fff0ef;color:#b42318}.tabla-card{padding:22px}.tabla-head{display:flex;justify-content:space-between;align-items:center}.tabla-head span{font-size:12px;color:#7b8497}.table-wrap{overflow-x:auto}table{width:100%;border-collapse:collapse;font-size:13px}th,td{text-align:left;padding:14px 10px;border-bottom:1px solid #edf0f5}th{color:#7b8497;font-size:11px;text-transform:uppercase;letter-spacing:.5px}.right{text-align:right}.delete{border:0;background:#fff0ef;color:#b42318;padding:7px 10px;border-radius:7px;cursor:pointer;font-size:11px}.empty{text-align:center;color:#7b8497;padding:28px}footer{text-align:center;color:#9aa2b2;font-size:12px;padding:28px}@media(max-width:760px){.layout{grid-template-columns:1fr}.resumen{grid-template-columns:1fr 1fr}.container{margin:24px auto}.filtros{grid-template-columns:1fr}.topbar{padding:0 18px}}

Overwriting static/estilos.css


##  Inicia servidor y abre la aplicación


In [36]:
import threading, time
from app import app, inicializar_db
from google.colab.output import eval_js

inicializar_db()
def ejecutar_flask():
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

hilo = threading.Thread(target=ejecutar_flask, daemon=True)
hilo.start()
time.sleep(2)
url = eval_js('google.colab.kernel.proxyPort(5000)')
print('Abre tu aplicación aquí:', url)

 * Serving Flask app 'app'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.


Abre tu aplicación aquí: https://5000-m-s-ase1c0-3dvd1gw5l22jg-c.asia-east1.prod.r.colab.dev


##
Funcionamiento
1. Inicia sesión con `admin` y `123`.
2. Registra fecha, descripción y monto de cada gasto.
3. Usa filtros Desde/Hasta para consultar un periodo.
4. Reporte con el botón **Descargar CSV**.
5. Para eliminar un gasto, usa el botón **Eliminar**

**Importante:** la información se guarda en `gastos.db` dentro de la sesión de Colab.